# **Aula 5**: Inferência Estatística

OBJETIVOS:

 - Aplicar testes estatísticos clássicos em um dataset, comparando grupos, avaliando normalidade e simulando cenários de teste A/B.

Usaremos o Data Warehouse construído na Aula 03: "dw_olist.csv"

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

In [ ]:
# Leitura do Data Warehouse (Star Schema)
#
# Esse arquivo foi gerado ao final da Aula 03

dw = pd.read_csv('dw_olist.csv')
dw.head()

In [ ]:
dw.describe()

## Métrica de interesse:
Valor do pedido (price)

In [ ]:
dw['price'].describe()

## Teste de Normalidade (Shapiro-Wilk)
Testes paramétricos exigem normalidade. Sempre verifique antes de escolher o teste.

- H0: os dados seguem distribuição normal
- H1: os dados NÃO seguem distribuição normal

In [ ]:
amostra = dw['price'].dropna().sample(500, random_state=42)

stat, p_valor = stats.shapiro(amostra)

print("p-valor:", p_valor)

if p_valor < 0.05:
    print("Rejeitamos H0: os dados NÃO são normais.")
else:
    print("Não rejeitamos H0: normalidade plausível.")

## TESTE DE NORMALIDADE - KOLMOGOROV-SMIRNOV (amostras grandes)

In [ ]:
media = dw['price'].mean()
desvio = dw['price'].std()

stat, p_valor = stats.kstest(
    dw['price'].dropna(),
    'norm',
    args=(media, desvio)
)

print("p-valor:", p_valor)

## Comparação de Dois Grupos – Paramétrico vs Não Paramétrico
Compare os p-valores. Se os dados não forem normais, o Mann-Whitney é mais confiável.

Pedidos com frete alto vs frete baixo

In [ ]:
limite_frete = dw['freight_value'].median()

grupo_frete_baixo = dw[dw['freight_value'] <= limite_frete]['price']
grupo_frete_alto = dw[dw['freight_value'] > limite_frete]['price']

print("Média frete baixo:", grupo_frete_baixo.mean())
print("Média frete alto:", grupo_frete_alto.mean())

t_stat, p_valor = stats.ttest_ind(
    grupo_frete_baixo,
    grupo_frete_alto,
    equal_var=False
)

print("p-valor:", p_valor)

## IGUALDADE DE VARIÂNCIAS (LEVENE)
- H0: as variâncias são iguais
- H1: pelo menos uma variância difere

In [ ]:
grupo_frete_baixo = dw[dw['freight_value'] <= dw['freight_value'].median()]['price']
grupo_frete_alto = dw[dw['freight_value'] > dw['freight_value'].median()]['price']

stat, p_valor = stats.levene(
    grupo_frete_baixo.dropna(),
    grupo_frete_alto.dropna()
)

print("p-valor:", p_valor)

if p_valor < 0.05:
    print("Variâncias diferentes → usar Welch ou teste não paramétrico.")
else:
    print("Variâncias semelhantes → teste t clássico é aceitável.")

## TESTE NÃO PARAMÉTRICO (MANN-WHITNEY U)

Usado quando normalidade é violada

In [ ]:
u_stat, p_valor = stats.mannwhitneyu(
    grupo_frete_baixo.dropna(),
    grupo_frete_alto.dropna(),
    alternative='two-sided'
)

print("p-valor:", p_valor)

In [ ]:
print("Diferença das médias:",
      grupo_frete_alto.mean() - grupo_frete_baixo.mean())

## Teste A/B

Cenário:
- Grupo A
- Grupo B

Métrica: price

In [ ]:
np.random.seed(42)

dw['grupo_ab'] = np.random.choice(['A', 'B'], size=len(dw))

grupo_A = dw[dw['grupo_ab'] == 'A']['price']
grupo_B = dw[dw['grupo_ab'] == 'B']['price']

print("Média grupo A:", grupo_A.mean())
print("Média grupo B:", grupo_B.mean())

t_stat, p_valor = stats.ttest_ind(
    grupo_A,
    grupo_B,
    equal_var=False
)

print("p-valor:", p_valor)

In [ ]:
if p_valor < 0.05:
    print("Diferença estatisticamente significativa.")
else:
    print("Não há evidência suficiente para mudança.")

## TESTE A/B - MÉTRICA BINÁRIA

Conversão simulada: pedido > mediana

In [ ]:
dw['conversao'] = (dw['price'] > dw['price'].median()).astype(int)

conv_controle = dw[dw['grupo_ab'] == 'Controle']['conversao']
conv_tratamento = dw[dw['grupo_ab'] == 'Tratamento']['conversao']

p1 = conv_controle.mean()
p2 = conv_tratamento.mean()

print("Conversão Controle:", p1)
print("Conversão Tratamento:", p2)

# Teste de proporções (aproximação normal)
z_stat, p_valor = stats.ttest_ind(conv_controle, conv_tratamento)

print("p-valor:", p_valor)